# Hands-On 9: Build an Advanced CrewAI Flow

### Definition
A **CrewAI Flow** is a stateful workflow that connects multiple steps, passes data between steps, and supports conditional routing.

### Goal
Create flow state, define steps, route conditionally, trigger crews, pass state, handle errors, and generate a final result.

### Flow
`Start → Research Crew → Quality Check → Analysis / Retry → Final Report`

In [1]:
from crewai import Agent, Task, Crew, Process
from crewai.flow.flow import Flow, listen, start
from pydantic import BaseModel, Field
from typing import List
import os
from dotenv import load_dotenv

load_dotenv(r"D:\Agent_Flow\.env", override=True)

print("OpenRouter key loaded:", bool(os.getenv("OPENROUTER_API_KEY")))
print("Model:", os.getenv("OPENAI_MODEL_NAME", "openai/gpt-4o-mini"))
print("Base URL:", os.getenv("OPENAI_BASE_URL"))

llm = os.getenv("OPENAI_MODEL_NAME", "openai/gpt-4o-mini")

OpenRouter key loaded: True
Model: openai/gpt-4o-mini
Base URL: https://openrouter.ai/api/v1


## 1. Define Flow State

The state stores the topic, research, quality, analysis, retries, errors, and final report.

In [2]:
class ResearchFlowState(BaseModel):
    topic: str = ""
    research_quality: str = ""
    research_output: str = ""
    analysis_output: str = ""
    final_report: str = ""
    errors: List[str] = Field(default_factory=list)
    retry_count: int = 0

print("Flow state created")

Flow state created


## 2. Create the Advanced Flow

The flow uses `@start()` for the first step and `@listen()` to connect later steps.

In [4]:
class ResearchFlow(Flow[ResearchFlowState]):

    @start()
    def initialize_topic(self):
        self.state.topic = "The Future of Quantum Computing in 2025"
        print("[Flow] Topic:", self.state.topic)
        return self.state.topic

    @listen(initialize_topic)
    async def conduct_research(self, topic):
        print("[Flow] Researching:", topic)

        researcher = Agent(
            role="Research Specialist",
            goal=f"Research: {topic}",
            backstory="An expert researcher who provides accurate findings.",
            llm=llm,
            verbose=False,
            allow_delegation=False
        )

        task = Task(
            description=(
                f"Research {topic}. Cover major developments, applications, "
                "challenges, opportunities, and future direction."
            ),
            expected_output="A structured research report with key findings.",
            agent=researcher
        )

        crew = Crew(
            agents=[researcher],
            tasks=[task],
            process=Process.sequential,
            verbose=False
        )

        try:
            result = await crew.kickoff_async()
            self.state.research_output = str(result)

            length = len(self.state.research_output)
            self.state.research_quality = (
                "high" if length > 500 else
                "medium" if length > 200 else
                "low"
            )

            print("[Flow] Quality:", self.state.research_quality)
            return self.state.research_quality

        except Exception as e:
            self.state.errors.append(f"Research error: {e}")
            self.state.research_quality = "low"
            return "low"

    @listen(conduct_research)
    async def conditional_analysis(self, quality):
        print("[Flow] Routing:", quality)

        if quality == "high":
            return await self._deep_analysis()

        if quality == "medium":
            return await self._standard_analysis()

        return await self._retry_research()

    async def _deep_analysis(self):
        analyst = Agent(
            role="Senior Analyst",
            goal="Provide deep strategic analysis",
            backstory="An expert analyst who identifies strategic implications.",
            llm=llm,
            verbose=False,
            allow_delegation=False
        )

        task = Task(
            description=(
                f"""Deeply analyze this research:

{self.state.research_output}

Cover implications, opportunities, risks, applications, and outlook."""
            ),
            expected_output="A detailed strategic analysis.",
            agent=analyst
        )

        crew = Crew(
            agents=[analyst],
            tasks=[task],
            verbose=False
        )

        try:
            result = await crew.kickoff_async()
            self.state.analysis_output = str(result)
            return "deep"

        except Exception as e:
            self.state.errors.append(f"Deep analysis error: {e}")
            self.state.analysis_output = "Deep analysis failed."
            return "failed"

    async def _standard_analysis(self):
        analyst = Agent(
            role="Business Analyst",
            goal="Provide a concise analysis",
            backstory="A practical analyst who converts research into insights.",
            llm=llm,
            verbose=False,
            allow_delegation=False
        )

        task = Task(
            description=f"""Analyze this research and identify the key findings:

{self.state.research_output}""",
            expected_output="A concise analysis with findings and recommendations.",
            agent=analyst
        )

        crew = Crew(
            agents=[analyst],
            tasks=[task],
            verbose=False
        )

        try:
            result = await crew.kickoff_async()
            self.state.analysis_output = str(result)
            return "standard"

        except Exception as e:
            self.state.errors.append(f"Standard analysis error: {e}")
            self.state.analysis_output = "Standard analysis failed."
            return "failed"

    async def _retry_research(self):
        self.state.retry_count += 1
        print("[Flow] Retry:", self.state.retry_count)

        if self.state.retry_count > 2:
            self.state.errors.append("Maximum research retries exceeded.")
            self.state.analysis_output = "Research failed after multiple attempts."
            return "failed"

        researcher = Agent(
            role="Senior Research Specialist",
            goal="Produce comprehensive research with specific examples",
            backstory="An expert researcher who provides detailed findings.",
            llm=llm,
            verbose=False,
            allow_delegation=False
        )

        task = Task(
            description=(
                f"Provide detailed research on {self.state.topic}. "
                "Include examples, developments, applications, challenges, and opportunities."
            ),
            expected_output="A comprehensive research report.",
            agent=researcher
        )

        crew = Crew(
            agents=[researcher],
            tasks=[task],
            verbose=False
        )

        try:
            result = await crew.kickoff_async()
            self.state.research_output = str(result)

            length = len(self.state.research_output)
            self.state.research_quality = (
                "high" if length > 500 else
                "medium" if length > 200 else
                "low"
            )

            return await self.conditional_analysis(
                self.state.research_quality
            )

        except Exception as e:
            self.state.errors.append(f"Retry error: {e}")
            return "failed"

    @listen(conditional_analysis)
    async def generate_final_report(self, analysis_type):
        print("[Flow] Generating final report:", analysis_type)

        if analysis_type == "failed":
            self.state.final_report = (
                "Report generation failed.\n"
                "Errors:\n" +
                "\n".join(self.state.errors)
            )
            return self.state.final_report

        writer = Agent(
            role="Report Writer",
            goal="Create a professional executive report",
            backstory="An expert writer who combines research and analysis.",
            llm=llm,
            verbose=False,
            allow_delegation=False
        )

        task = Task(
            description=(
                f"""Create a professional report on {self.state.topic}.

Research:
{self.state.research_output}

Analysis:
{self.state.analysis_output}

Include an executive summary, findings, implications, challenges, 
opportunities, and conclusion."""
            ),
            expected_output="A polished executive report.",
            agent=writer
        )

        crew = Crew(
            agents=[writer],
            tasks=[task],
            verbose=False
        )

        try:
            result = await crew.kickoff_async()
            self.state.final_report = str(result)

        except Exception as e:
            self.state.errors.append(f"Final report error: {e}")
            self.state.final_report = f"Report generation failed: {e}"

        return self.state.final_report


print("ResearchFlow created")

ResearchFlow created


## 3. Execute the Flow

The notebook uses async execution because CrewAI crews are executed asynchronously in the VS Code/Jupyter environment.

In [5]:
print("=" * 70)
print("EXECUTING ADVANCED CREWAI FLOW")
print("=" * 70)

flow = ResearchFlow()
final_report = await flow.kickoff_async()

print("\n" + "=" * 70)
print("FLOW EXECUTION COMPLETE")
print("=" * 70)
print(final_report)

EXECUTING ADVANCED CREWAI FLOW


╭─────────────────────────────────────────────── 🌊 Flow Execution ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Starting Flow Execution                                                                                        │
│  Name: ResearchFlow                                                                                             │
│  ID: ff5ea344-4b31-4abf-94a8-8d0dc0b5992e                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 🌊 Flow Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Flow Started                                                                                                   │
│  Name: ResearchFlow                                                                                             │
│  ID: ff5ea344-4b31-4abf-94a8-8d0dc0b5992e                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── 🔄 Flow Method Running ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: initialize_topic                                                                                       │
│  Status: Running                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Flow] Topic: The Future of Quantum Computing in 2025


[Flow] Researching: The Future of Quantum Computing in 2025


... (7 outputs left out)

╭──────────────────────────────────────────── 🔄 Flow Method Running ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: generate_final_report                                                                                  │
│  Status: Running                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── ✅ Flow Method Completed ────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: generate_final_report                                                                                  │
│  Status: Completed                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── ✅ Flow Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Flow Execution Completed                                                                                       │
│  Name: ResearchFlow                                                                                             │
│  ID: ff5ea344-4b31-4abf-94a8-8d0dc0b5992e                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


FLOW EXECUTION COMPLETE
# Executive Report: The Future of Quantum Computing in 2025

## Executive Summary
As we transition towards 2025, quantum computing is positioned as a transformative technology with the potential to revolutionize diverse sectors, from pharmaceuticals to finance and logistics. This report collates insights and forecasts regarding major advancements, applications, challenges, and opportunities in the quantum computing landscape. By 2025, enhanced hardware, a robust software ecosystem, and significant contributions to various industries will reshape our computing paradigms, though several challenges and risks will need to be addressed for successful implementation.

## 1. Introduction to Quantum Computing
Quantum computing capitalizes on the principles of quantum mechanics to process information in ways that diverge sharply from classical computing. At its core, quantum computing utilizes qubits, which can exist in various states simultaneously, allowing for calcul

## 4. Inspect Flow State

In [6]:
print("Topic:", flow.state.topic)
print("Research Quality:", flow.state.research_quality)
print("Retry Count:", flow.state.retry_count)
print("Errors:", flow.state.errors)
print("Research Length:", len(flow.state.research_output))
print("Analysis Length:", len(flow.state.analysis_output))
print("Final Report Length:", len(flow.state.final_report))

Topic: The Future of Quantum Computing in 2025
Research Quality: high
Retry Count: 0
Errors: []
Research Length: 6511
Analysis Length: 6756
Final Report Length: 12540


## 5. Final Result

In [7]:
print("=" * 70)
print("FINAL REPORT")
print("=" * 70)
print(flow.state.final_report)

FINAL REPORT
# Executive Report: The Future of Quantum Computing in 2025

## Executive Summary
As we transition towards 2025, quantum computing is positioned as a transformative technology with the potential to revolutionize diverse sectors, from pharmaceuticals to finance and logistics. This report collates insights and forecasts regarding major advancements, applications, challenges, and opportunities in the quantum computing landscape. By 2025, enhanced hardware, a robust software ecosystem, and significant contributions to various industries will reshape our computing paradigms, though several challenges and risks will need to be addressed for successful implementation.

## 1. Introduction to Quantum Computing
Quantum computing capitalizes on the principles of quantum mechanics to process information in ways that diverge sharply from classical computing. At its core, quantum computing utilizes qubits, which can exist in various states simultaneously, allowing for calculations at sp

## 6. Final Verification

- ✅ Flow state created
- ✅ Workflow steps defined
- ✅ Conditional routing added
- ✅ Crew execution triggered
- ✅ State passed between steps
- ✅ Retry logic added
- ✅ Error handling added
- ✅ Final result generated

### Architecture

```text
START
  ↓
Initialize Topic
  ↓
Research Crew
  ↓
Quality Check
  ├── HIGH → Deep Analysis ─────┐
  ├── MEDIUM → Standard Analysis ┤
  └── LOW → Retry Research ─────┘
                                  ↓
                           Final Report
                                  ↓
                                 END
```